# RSNA Breast Baseline - Inference

This notebook shows a simple inference pipeline based on the pregenerated datasets.

In [ ]:
DEBUG = False

## Initialization

In [ ]:
try:
    import pylibjpeg
except:
    !pip install /kaggle/input/rsna-2022-whl/{pydicom-2.3.0-py3-none-any.whl,pylibjpeg-1.4.0-py3-none-any.whl,python_gdcm-3.0.15-cp37-cp37m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl}

In [ ]:
import os
import sys
import cv2
import glob
import gdcm
import json
import pydicom
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
from joblib import Parallel, delayed

## Data Preparation

- I use the same strategy as in https://www.kaggle.com/code/theoviel/dicom-resized-png-jpg

In [ ]:
test_images = glob.glob("/kaggle/input/rsna-breast-cancer-detection/test_images/*/*.dcm")

if DEBUG:
    test_images = glob.glob("/kaggle/input/rsna-breast-cancer-detection/train_images/10042/*.dcm")
    
print("Number of images :", len(test_images))

In [ ]:
SAVE_FOLDER = "/kaggle/tmp/output/"
SIZE = 512
EXTENSION = "png"

os.makedirs(SAVE_FOLDER, exist_ok=True)

In [ ]:
def process(f, size=512, save_folder="", extension="png"):
    patient = f.split('/')[-2]
    image = f.split('/')[-1][:-4]

    dicom = pydicom.dcmread(f)
    img = dicom.pixel_array

    img = (img - img.min()) / (img.max() - img.min())

    if dicom.PhotometricInterpretation == "MONOCHROME1":
        img = 1 - img

    img = cv2.resize(img, (size, size))

    cv2.imwrite(save_folder + f"{patient}_{image}.{extension}", (img * 255).astype(np.uint8))

In [ ]:
_ = Parallel(n_jobs=4)(
    delayed(process)(uid, size=SIZE, save_folder=SAVE_FOLDER, extension=EXTENSION)
    for uid in tqdm(test_images)
)

## Inference

### Utils

### Main

In [ ]:
df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/test.csv")
df['cancer'] = 0
df['path'] = SAVE_FOLDER + df["patient_id"].astype(str) + "_" + df["image_id"].astype(str) + ".png"

In [ ]:
df

In [ ]:
EXP_FOLDERS = [
    "/kaggle/input/rsna-breast-weights-public/"
]

### Main loop

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms as T

to_tensor = T.Compose([
    T.ToTensor(),
    T.Resize((256, 256)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#     T.Normalize(mean=[0.66437738, 0.50478148, 0.70114894], std=[0.15825711, 0.24371008, 0.13832686])
])


In [ ]:
import yaml
from torchvision import models
import torch
import torch.nn as nn
from torchvision import transforms as T
from PIL import Image
BACKBONE_MAPPING = {
    'b0': {
        'm': models.efficientnet_b0,
        'last_layer': 'classifier',
        'in_features': 1280
    },
    'b1': {
        'm': models.efficientnet_b0,
        'last_layer': 'classifier',
        'in_features': 1280
    },
    'resnet18': {
        'm': models.resnet18,
        'last_layer': 'fc',
        'in_features': 512
    },
    'resnet50': {
        'm': models.resnet50,
        'last_layer': 'fc',
        'in_features': 2048
    }
}

def get_module_list(config):
    model_list = []
    for name, arg_list in config:
        template = getattr(nn, name)
        m = template(*arg_list)
        model_list.append(m)
    return model_list

class HEMaxBlock(nn.Module):
    def __init__(self, beta):
        super(HEMaxBlock, self).__init__()
        self.beta = beta
    
    def forward(self, X):
        '''
            X: (batch_size, channel, w, h)
        '''
        shape = X.shape
        
        X_mask = X.clone()
        max_value, _ = torch.max(X_mask.view(*shape[:2], -1), dim = -1)
        mask = X_mask == max_value[:, :, None, None]
        
        one = torch.ones_like(X)
        one[mask] *= self.beta
        X = X * one
        return X

class DynamicHE(nn.Module):
    def __init__(self, backbone, cut_layer, bottle_layers, head_layers, beta):
        super().__init__()
        net_info = BACKBONE_MAPPING.get(backbone)
        net = net_info['m'](pretrained=False)
        blocks = []
        for name, layer in net.named_children():
            blocks.append(layer)
            if name == cut_layer[0]:
                self.block_pre = nn.Sequential(*blocks)
                blocks = []
            if name == cut_layer[1]:
                self.block_pos = nn.Sequential(*blocks)
                break
        self.bottle_layers = nn.Sequential(*get_module_list(bottle_layers))
        self.head = nn.ModuleList(
            nn.Sequential(*get_module_list(head))
            for head in head_layers
        )
        self.n_head = len(head_layers)
        self.he_block = HEMaxBlock(beta)

    def forward(self, X):
        '''
            X: (batch_size, channel, w, h)
        '''
        out = self.block_pre(X)
        if self.training:
            out = self.he_block(out)
        out = self.block_pos(out)
        out = nn.functional.adaptive_avg_pool2d(out, (1, 1))
        out = out.flatten(1)
        out = self.bottle_layers(out)
        out_head = []
        for i in range(self.n_head):
            out_head.append(self.head[i](out))

        return out_head
    


class Cls:
    def __init__(self, model_cf):
        self.device = model_cf['device']
        self.model = DynamicHE(**model_cf['args'])
        self.model.to(device=self.device)
        checkpoint = torch.load(model_cf['weight'], map_location=self.device)
        self.model.load_state_dict(checkpoint['model'])
        self.model.eval()

        self.to_tensor = T.Compose([
                                    T.ToTensor(),
                                    T.Resize((256,256)),
                                    T.Normalize(mean=[0.485, 0.456, 0.406], 
                                                std=[0.229, 0.224, 0.225])
                                ])
    def __call__(self, imgs):
        imgs = torch.stack([self.to_tensor(img) for img in imgs]).to(device = self.device)
        with torch.no_grad():
            outputs = self.model(imgs)
        
        pred_label = [torch.softmax(output, dim = 1).cpu().tolist() for output in outputs]
        
        return [np.argmax(p).item() for p in pred_label[0]]
        

In [ ]:
config={'weight': '/kaggle/input/weights/resnet50_cut256_2cls.pt', 
        'device': 'cuda', 
        'args': {
            'backbone': 'resnet50', 
            'beta': 0.5, 
            'bottle_layers': [
                ['Linear', [2048, 512]], 
                ['BatchNorm1d', [512]],
                ['ReLU', []],
                ['Dropout', [0.5]]
            ], 
            'cut_layer': ['layer4', 'layer4'], 
            'head_layers': [
                [['Linear', [512, 2]]], # cancer
                #[['Linear', [512, 2]]], # laterality
                #[['Linear', [512, 2]]], # view
                #[['Linear', [512, 2]]], # invasive
                #[['Linear', [512, 3]]], # BIRADS
                #[['Linear', [512, 2]]], # implant
                [['Linear', [512, 4]]]] # density
        } 
       }

In [ ]:
model = Cls(config)

In [ ]:
path = df.path.to_list()
laterality = df.laterality.to_list()
batch_size = 32
pred = []
for idx in range(0, len(path), batch_size):
#     imgs = [Image.open(p).convert('RGB') for p in path[idx:idx + batch_size]]
    imgs = []
    for p, l in zip(path[idx:idx + batch_size], laterality[idx:idx + batch_size]):
        img = Image.open(p).convert('RGB')
        width, height= img.size

        if l == 'L':
            left = 0
            right = int(width / 3 * 2)
        else:
            left = int(width / 3)
            right = width
        croped_img = img.crop((left, 0, right, height))
        imgs.append(croped_img)

    out = model(imgs)
    pred.extend(out)    

In [ ]:
pred, path

In [ ]:
# sub = pd.DataFrame({
#     'prediction_id': df.prediction_id,
#     'cancer': pred
# })

df['cancer'] = pred

sub = df[['prediction_id', 'cancer']].groupby("prediction_id").mean().reset_index()
# sub.to_csv('submission.csv', index=False)
sub.to_csv('/kaggle/working/submission.csv', index=False)

sub.head()

Done !